In [1]:
import pandas as pd
import numpy as np
import re
import os
from tqdm.auto import tqdm

# =============================================================================
# 1. 설정 및 정규식 정의
# =============================================================================

SECTION_MAPPING = {
    "AUDIT": ["감사의견_등급", "감사의견_전문"],
    "MD&A":  ["MD&A"],
    "ETC":   ["기타_보호사항"]
}

ID_CANDIDATES = ["corp_code", "기업코드", "종목코드"]
YEAR_CANDIDATES = ["target_year", "bs_year", "year", "사업연도", "결산년도"]
NAME_CANDIDATES = ["corp_name", "기업명", "회사명"]

# 로마자 변환 맵
ROMAN_MAP = str.maketrans({
    "Ⅰ":"I","Ⅱ":"II","Ⅲ":"III","Ⅳ":"IV","Ⅴ":"V","Ⅵ":"VI",
    "Ⅶ":"VII","Ⅷ":"VIII","Ⅸ":"IX","Ⅹ":"X"
})
# 허용 문자 정규식
ALLOWED_RE = re.compile(r"[^가-힣ㄱ-ㅎㅏ-ㅣA-Za-z0-9\s\[\]<>/.,\-()٪%:;'\"!?&+=]")

# =============================================================================
# 2. 전처리 함수 정의
# =============================================================================

def smart_mask_numbers(text: str) -> str:
    """
    금액/비율은 보존하고 일반 숫자만 마스킹
    """
    if not text: return ""
    
    # 금융/비율 숫자 보호 (숫자 + 단위)
    financial_pattern = re.compile(r'(\d+(?:[,\.]\d+)*)(\s*(?:원|천원|백만원|억원|조원|달러|유로|엔|%|퍼센트|pt))')
    protected_map = {}
    
    def protect_match(match):
        key = f"{{{{FIN_NUM_{len(protected_map)}}}}}"
        protected_map[key] = match.group(1) + match.group(2) 
        return key

    text = financial_pattern.sub(protect_match, text)

    # 연도 마스킹
    text = re.sub(r"\b(19|20)\d{2}\b", "__YEAR__", text)
    # 일반 숫자 마스킹
    text = re.sub(r"\d+(?:,\d+)?(?:\.\d+)?", "__NUM__", text)

    # 복원
    for key, val in protected_map.items():
        text = text.replace(key, val)
        
    return text

def normalize_text(t, mask_nums=True) -> str:
    if t is None or (isinstance(t, float) and np.isnan(t)):
        return ""
    
    t = str(t).translate(ROMAN_MAP)
    t = re.sub(r"</?[a-zA-Z][^>]*>", " ", t)  # HTML 태그 제거
    t = re.sub(r"[-_=]{3,}", " ", t)         # 구분선 제거
    
    if mask_nums:
        t = smart_mask_numbers(t)
        
    t = ALLOWED_RE.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()

def mask_company_name(text: str, corp_name: str) -> str:
    if not corp_name or (isinstance(corp_name, float) and np.isnan(corp_name)):
        return text
    name = str(corp_name).strip()
    patterns = [re.escape(name), re.escape(name.replace(" ", ""))]
    for p in patterns:
        text = re.sub(p, "<COMPANY>", text)
    return text

# =============================================================================
# 3. 데이터 로드 및 처리
# =============================================================================

def read_csv_safely(path: str) -> pd.DataFrame:
    # 로컬 경로 문제 방지용 절대 경로 변환 (선택 사항)
    if not os.path.exists(path):
        print(f"⚠️ 경고: 파일을 찾을 수 없습니다 -> {path}")
        return pd.DataFrame() # 빈 데이터프레임 반환

    encodings = ["utf-8-sig", "utf-8", "cp949"]
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except:
            continue
    raise RuntimeError(f"CSV 로드 실패 (인코딩 확인 필요): {path}")

def process_and_split(files, is_default, section_lists):
    for file_path in tqdm(files, desc=f"{'부도' if is_default else '정상'}기업 처리"):
        df = read_csv_safely(file_path)
        if df.empty: continue

        try:
            id_col = next((c for c in ID_CANDIDATES if c in df.columns), "corp_code")
            year_col = next((c for c in YEAR_CANDIDATES if c in df.columns), "target_year")
            name_col = next((c for c in NAME_CANDIDATES if c in df.columns), None)
            
            if year_col not in df.columns:
                df[year_col] = np.nan 
            
            df["y"] = 1 if is_default else 0
            
            for section_name, target_cols in SECTION_MAPPING.items():
                texts = []
                for _, row in df.iterrows():
                    row_texts = []
                    corp_name = row.get(name_col, "") if name_col else ""
                    
                    for col in target_cols:
                        if col in df.columns:
                            val = normalize_text(row[col], mask_nums=True)
                            val = mask_company_name(val, corp_name)
                            if val:
                                row_texts.append(f"[{col}] {val}")
                    
                    full_text = " ".join(row_texts)
                    texts.append(full_text)
                
                temp_df = pd.DataFrame({
                    "corp_code": df[id_col],
                    "target_year": df[year_col],
                    "y": df["y"],
                    "text": texts
                })
                section_lists[section_name].append(temp_df)
                
        except Exception as e:
            print(f"❌ 에러 발생 ({file_path}): {e}")

# =============================================================================
# 4. 실행부
# =============================================================================

# 현재 파이썬 파일이 있는 위치를 기준으로 경로 설정
base_path = "."  # 현재 폴더

files_normal = [
    os.path.join(base_path, "정상기업_크롤링_Batch_1_2015.csv"),
    os.path.join(base_path, "정상기업_크롤링_Batch_2_new.csv"),
    os.path.join(base_path, "정상기업_크롤링_Batch_3_new.csv"),
    os.path.join(base_path, "정상기업_크롤링_Batch_4_2015.csv"),
    os.path.join(base_path, "정상기업_크롤링_Batch_5_new.csv"),
    os.path.join(base_path, "정상기업_크롤링_Batch_6_new.csv")
]

files_bankrupt = [
    os.path.join(base_path, "부도기업_진짜다시.csv")
]

data_storage = {"AUDIT": [], "MD&A": [], "ETC": []}

print("🚀 로컬 환경에서 데이터 전처리 시작...")

process_and_split(files_normal, is_default=False, section_lists=data_storage)
process_and_split(files_bankrupt, is_default=True, section_lists=data_storage)

print("\n💾 파일 저장 중...")

for section, dfs in data_storage.items():
    if dfs:
        final_df = pd.concat(dfs, ignore_index=True)
        filename = f"dataset_{section.replace('&', '')}.csv"
        
        # 파일 저장
        final_df.to_csv(filename, index=False, encoding="utf-8-sig")
        
        print(f"\n[{section}] 저장 완료 -> {filename}")
        print(f"  - 총 샘플 수: {len(final_df):,}개")
        print(f"  - 예시: {final_df['text'].iloc[0][:50]}...")
    else:
        print(f"\n⚠️ [{section}] 데이터가 없습니다.")

print("\n✨ 작업 완료! 같은 폴더에 dataset_*.csv 파일이 생성되었습니다.")

🚀 로컬 환경에서 데이터 전처리 시작...


정상기업 처리:   0%|          | 0/6 [00:00<?, ?it/s]

부도기업 처리:   0%|          | 0/1 [00:00<?, ?it/s]


💾 파일 저장 중...

[AUDIT] 저장 완료 -> dataset_AUDIT.csv
  - 총 샘플 수: 28,222개
  - 예시: [감사의견_등급] 적정 [감사의견_전문] V. 회계감사인의 감사의견 등 NUM . 외부감사...

[MD&A] 저장 완료 -> dataset_MDA.csv
  - 총 샘플 수: 28,222개
  - 예시: [MD&A] NUM . 예측정보에 대한 주의사항 본 자료는 미래에 대한 '예측정보'를 포함...

[ETC] 저장 완료 -> dataset_ETC.csv
  - 총 샘플 수: 28,222개
  - 예시: [기타_보호사항] NUM . 공시내용 진행 및 변경사항 NUM . 우발부채 등에 관한 사항...

✨ 작업 완료! 같은 폴더에 dataset_*.csv 파일이 생성되었습니다.


In [2]:
AUDIT = pd.read_csv('dataset_AUDIT.csv')

AUDIT.head()


,corp_code,target_year,y,text
0,126380,2024,0,[감사의견_등급] 적정 [감사의견_전문] V. 회계감사인의 감사의견 등 NUM . ...
1,126380,2023,0,[감사의견_등급] 적정 [감사의견_전문] V. 회계감사인의 감사의견 등 NUM . ...
2,126380,2022,0,[감사의견_등급] 한정 [감사의견_전문] V. 회계감사인의 감사의견 등 NUM . ...
3,126380,2021,0,[감사의견_등급] 한정 [감사의견_전문] V. 회계감사인의 감사의견 등 NUM . ...
4,126380,2020,0,[감사의견_등급] 한정 [감사의견_전문] V. 감사인의 감사의견 등 NUM . 회계...


In [3]:
ETC = pd.read_csv('dataset_AUDIT.csv')

ETC.head()

,corp_code,target_year,y,text
0,126380,2024,0,[감사의견_등급] 적정 [감사의견_전문] V. 회계감사인의 감사의견 등 NUM . ...
1,126380,2023,0,[감사의견_등급] 적정 [감사의견_전문] V. 회계감사인의 감사의견 등 NUM . ...
2,126380,2022,0,[감사의견_등급] 한정 [감사의견_전문] V. 회계감사인의 감사의견 등 NUM . ...
3,126380,2021,0,[감사의견_등급] 한정 [감사의견_전문] V. 회계감사인의 감사의견 등 NUM . ...
4,126380,2020,0,[감사의견_등급] 한정 [감사의견_전문] V. 감사인의 감사의견 등 NUM . 회계...


In [6]:
MDA = pd.read_csv('dataset_AUDIT.csv')

MDA.head()

,corp_code,target_year,y,text
0,126380,2024,0,[감사의견_등급] 적정 [감사의견_전문] V. 회계감사인의 감사의견 등 NUM . ...
1,126380,2023,0,[감사의견_등급] 적정 [감사의견_전문] V. 회계감사인의 감사의견 등 NUM . ...
2,126380,2022,0,[감사의견_등급] 한정 [감사의견_전문] V. 회계감사인의 감사의견 등 NUM . ...
3,126380,2021,0,[감사의견_등급] 한정 [감사의견_전문] V. 회계감사인의 감사의견 등 NUM . ...
4,126380,2020,0,[감사의견_등급] 한정 [감사의견_전문] V. 감사인의 감사의견 등 NUM . 회계...
